# Commerce CSV to Delta

Attach a **schema-enabled project lakehouse** as default. Upload the six CSVs to `Files/commerce`. This notebook has not yet been executed in a Fabric tenant. See `FABRIC_SETUP.md`.

The CSVs are the raw layer; typed, validated DataFrames form the transformation layer; the six Delta outputs form the reporting model. No production deployment is implied.

## 1. Read explicit types and remove exact duplicates

In [ ]:
from functools import reduce
from pyspark.sql import functions as F

BASE_PATH = "Files/commerce"
# Use a dedicated project lakehouse. True replaces only the six named project tables.
REPLACE_PROJECT_TABLES = False

schemas = {
    "dim_customer": "customer_key INT, customer_label STRING, city STRING, region STRING",
    "dim_product": "product_key INT, product_label STRING, category STRING, list_price_inr DECIMAL(18,2)",
    "dim_warehouse": "warehouse_key INT, warehouse_label STRING, city STRING, region STRING",
    "dim_date": "date_key INT, date DATE, year INT, month_number INT, month_name STRING, day_of_week INT",
    "fact_sales": "sales_line_key STRING, order_id STRING, line_number INT, order_date_key INT, customer_key INT, product_key INT, warehouse_key INT, order_status STRING, quantity INT, unit_price_inr DECIMAL(18,2), discount_inr DECIMAL(18,2), sales_before_refunds_inr DECIMAL(18,2), refund_inr DECIMAL(18,2), net_sales_inr DECIMAL(18,2)",
    "fact_shipments": "order_id STRING, order_date_key INT, customer_key INT, warehouse_key INT, promised_date DATE, delivered_date DATE, on_time_flag INT, delivery_days INT",
}
keys = {"dim_customer": "customer_key", "dim_product": "product_key",
        "dim_warehouse": "warehouse_key", "dim_date": "date_key",
        "fact_sales": "sales_line_key", "fact_shipments": "order_id"}
tables = {}
load_counts = []

for name, schema in schemas.items():
    raw = (spark.read.option("header", True).option("mode", "FAILFAST")
           .option("enforceSchema", False).option("dateFormat", "yyyy-MM-dd")
           .schema(schema).csv(f"{BASE_PATH}/{name}.csv"))
    raw_count = raw.count()
    clean = raw.dropDuplicates()  # identical rows only; conflicting keys are rejected below
    clean_count = clean.count()
    if name != "fact_shipments" and clean_count == 0:
        raise ValueError(f"{name}: unexpected empty table")
    nulls = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in clean.columns])
    if clean.filter(nulls).limit(1).count():
        raise ValueError(f"{name}: missing required values or failed type conversions")
    if clean.groupBy(keys[name]).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{name}: conflicting duplicate primary key")
    tables[name] = clean
    load_counts.append((name, raw_count, clean_count, raw_count - clean_count))

display(spark.createDataFrame(load_counts, ["table", "source_rows", "clean_rows", "exact_duplicates_removed"]))


## 2. Validate dimensions, order grain, money, and delivery rules

Any failed check stops execution before writes. Financial totals should reconcile to `manifest.json`.

In [ ]:
def require_empty(df, message):
    if df.limit(1).count():
        raise ValueError(message)

for fact in ("fact_sales", "fact_shipments"):
    pairs = [("customer_key", "dim_customer", "customer_key"),
             ("warehouse_key", "dim_warehouse", "warehouse_key"),
             ("order_date_key", "dim_date", "date_key")]
    if fact == "fact_sales":
        pairs.append(("product_key", "dim_product", "product_key"))
    for foreign, dim, primary in pairs:
        members = tables[dim].select(F.col(primary).alias(foreign))
        require_empty(tables[fact].join(members, foreign, "left_anti"),
                      f"{fact}: orphan {foreign}")

sales = tables["fact_sales"]
shipments = tables["fact_shipments"]
require_empty(sales.filter(~F.col("order_status").isin("COMPLETED", "RETURNED", "CANCELLED")), "Invalid order status")
require_empty(sales.filter((F.col("quantity") <= 0) | (F.col("unit_price_inr") < 0) |
                          (F.col("discount_inr") < 0) |
                          (F.col("discount_inr") > F.col("quantity") * F.col("unit_price_inr"))), "Invalid quantity, price, or discount")
expected_sales = F.when(F.col("order_status") == "CANCELLED", F.lit(0)).otherwise(
    F.col("quantity") * F.col("unit_price_inr") - F.col("discount_inr"))
expected_refund = F.when(F.col("order_status") == "RETURNED", F.col("sales_before_refunds_inr")).otherwise(F.lit(0))
require_empty(sales.filter((F.col("sales_before_refunds_inr") != expected_sales) |
                          (F.col("refund_inr") != expected_refund) |
                          (F.col("net_sales_inr") != F.col("sales_before_refunds_inr") - F.col("refund_inr"))), "Financial reconciliation failed")

# All lines in an order must agree on its status, customer, warehouse, and order date.
orders = sales.select("order_id", "order_status", "customer_key", "warehouse_key", "order_date_key").distinct()
require_empty(orders.groupBy("order_id").count().filter(F.col("count") != 1), "Inconsistent order header")
fulfilled = orders.filter(F.col("order_status") != "CANCELLED").drop("order_status")
shipment_headers = shipments.select("order_id", "customer_key", "warehouse_key", "order_date_key")
header_columns = ["order_id", "customer_key", "warehouse_key", "order_date_key"]
require_empty(fulfilled.join(shipment_headers, header_columns, "left_anti"), "Missing or mismatched shipment")
require_empty(shipment_headers.join(fulfilled, header_columns, "left_anti"), "Unexpected or mismatched shipment")
shipment_dates = shipments.join(tables["dim_date"].select(F.col("date_key").alias("order_date_key"), F.col("date").alias("order_date")), "order_date_key")
expected_on_time = F.when(F.col("delivered_date") <= F.col("promised_date"), 1).otherwise(0)
require_empty(shipment_dates.filter((F.col("on_time_flag") != expected_on_time) |
                                   (F.col("delivered_date") < F.col("order_date")) |
                                   (F.col("promised_date") < F.col("order_date")) |
                                   (F.col("delivery_days") != F.datediff("delivered_date", "order_date"))), "Invalid shipment dates or metrics")

print("All integrity checks passed. No output tables have been written yet.")
display(sales.agg(F.sum("net_sales_inr").alias("net_sales_inr"), F.countDistinct("order_id").alias("orders_placed")))


## 3. Write the reporting tables

Replacement is off by default. These are separate table writes, not an atomic multi-table transaction. If a write fails, resolve it and rerun the whole notebook in the project lakehouse with deliberate replacement enabled.

In [ ]:
# Preflight every destination so a routine rerun fails before the first write.
existing = [name for name in tables if spark.catalog.tableExists(f"dbo.{name}")]
if existing and not REPLACE_PROJECT_TABLES:
    raise ValueError(f"Tables already exist: {existing}. Use a fresh project lakehouse or explicitly enable replacement.")

mode = "overwrite" if REPLACE_PROJECT_TABLES else "errorifexists"
for name, df in tables.items():
    (df.write.format("delta").mode(mode)
     .option("overwriteSchema", "true").saveAsTable(f"dbo.{name}"))
    print(f"Saved dbo.{name}")

print("Load complete. Run analytics.sql in the SQL analytics endpoint after synchronization.")
